In [0]:
# Install necessary libraries
%pip install geopandas
%pip install openpyxl
%pip install matplotlib

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Load your bird data
df = pd.read_excel(
   '/Volumes/prd_dash_lab/aphw_wbm_unrestricted/wild_bird_mortality/WildBirdMortality_010423_110925_UKFC_EJ_LIMS.xlsx',
    engine='openpyxl'
)

# Filter for months May–September
months = ["May", "June", "July", "August", "September"]
df = df[df['ReportMonth'].isin(months)].copy()

# Filter for species containing keywords
def filter_species_by_keywords(df):
    keywords = ['corvid', 'song bird', 'bird of prey']
    return df[df['ReportSpecies'].str.contains('|'.join(keywords), case=False, na=False)].copy()

df = filter_species_by_keywords(df)

# Drop rows with missing coordinates
df = df.dropna(subset=['Dlong', 'Dlat'])

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Dlong'], df['Dlat']),
    crs="EPSG:4326"
)

# Load counties/unitary authorities boundaries from DASH
try:
    counties_gdf = gpd.read_file(
      '/dbfs/mnt/base/unrestricted/source_ons_open_geography_portal/dataset_counties_uni_authorities_bdy_uk_bfe/format_SHP_counties_uni_authorities_bdy_uk_bfe/LATEST_counties_uni_authorities_bdy_uk_bfe/'
    )
except (IOError, OSError) as e:
    print(f"Error reading shapefile: {e}")

# Ensure both are in the same CRS
if 'counties_gdf' in locals():
    gdf = gdf.to_crs(counties_gdf.crs)

    # Use sjoin function to assign county to each bird location
    county_col = next((col for col in counties_gdf.columns if 'NM' in col or 'name' in col.lower()), None)
    if county_col:
        result = gpd.sjoin(
            gdf, 
            counties_gdf[['geometry', county_col]],  
            how='left',
            lsuffix='_caller',
            rsuffix='_other'
        )
        result = result.rename(columns={county_col: 'county'})

        # Get the count by county
        count_by_county = result['county'].value_counts().reset_index()
        count_by_county.columns = ['County', 'Count']

        # Display the count by county
        display(count_by_county)

        # Get the county counts by month
        county_counts_by_month = result.groupby(['ReportMonth', 'county']).size().reset_index(name='count')

        # Display the county counts by month
        display(county_counts_by_month)

        # Convert the result to a Pandas DataFrame
        county_counts_by_month_df = county_counts_by_month

        # Download the result to Excel
        import pandas as pd
        county_counts_by_month_df.to_excel("county_counts_by_month.xlsx", index=False)

        # Get the count by county broken down by species type
        count_by_county_species = result.groupby(['county', 'ReportSpecies']).size().reset_index(name='count')

        # Display the count by county broken down by species type
        display(count_by_county_species)

        # Filter out rows with null 'county' values
        joined_ = result[~result['county'].isna()]

        # Get rows with null 'county' values
        nulls = result[result['county'].isna()]

        # Use sjoin_nearest function to assign county to each null location
        near_nulls = gdf.sjoin_nearest(counties_gdf, how='left', distance_col='distance', max_distance=2500) #2.5 km from shore
        near_nulls['county']= near_nulls ['CTYUA24NM']
        near_nulls = near_nulls ['county'].fillna('At Sea')
        data = pd.concat([joined_['county'], near_nulls], ignore_index=True)
        display(data)

# Create a separate choropleth map for each month
months = result['ReportMonth'].unique()
for month in months:
    month_data = result[result['ReportMonth'] == month]
    month_counts = month_data['county'].value_counts().reset_index()
    month_counts.columns = ['County', 'Count']

    # Create a choropleth map
    plt.figure(figsize=(40, 40))  # Increased figure size
    merged_gdf = counties_gdf.merge(month_counts, left_on=county_col, right_on='County', how='left')
    ax = merged_gdf.plot(column='Count', cmap='Blues', legend=True)  # Added legend
    ax.set_title(f'{month} County Counts')  # Added title
    ax.set_xlabel('Longitude')  # Added x-axis label
    ax.set_ylabel('Latitude')  # Added y-axis label
    plt.savefig(f"{month}_county_counts.png")

    # Create a point-based map with different colors for species
    plt.figure(figsize=(40, 40))  # Increased figure size
    month_data['species_category'] = pd.Categorical(month_data['ReportSpecies'])
    ax = month_data.plot(column='species_category', categorical=True, cmap='tab20', legend=True)  # Added legend
    ax.set_title(f'{month} Species Distribution')  # Added title
    ax.set_xlabel('Longitude')  # Added x-axis label
    ax.set_ylabel('Latitude')  # Added y-axis label
    plt.savefig(f"{month}_species_distribution.png")

# Create a plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,1,figsize=(50, 50))  # Increased figure size

# Plot only the outlines of the shapes
counties_gdf.plot(ax=ax, facecolor='none', edgecolor='blue')
if 'nulls' in locals():
    nulls['species_category'] = pd.Categorical(nulls['ReportSpecies'])
    nulls.plot(ax=ax, column='species_category', categorical=True, cmap='tab20', markersize=10, legend=True)  # Added legend
ax.set_title('Counties Plot')  # Added title
ax.set_xlabel('Longitude')  # Added x-axis label
ax.set_ylabel('Latitude')  # Added y-axis label
ax.axis('on')  # Turned axes on

# Save the plot to a PNG file
plt.savefig("counties_plot.png", bbox_inches='tight', dpi=300)

In [0]:
import matplotlib.pyplot as plt
import geopandas as gpd

# Create a choropleth map
combined_data = result
combined_counts = combined_data['county'].value_counts().reset_index()
combined_counts.columns = ['County', 'Count']

# Create a choropleth map
plt.figure(figsize=(45, 45))  # Increased figure size
merged_gdf = counties_gdf.merge(combined_counts, left_on='CTYUA24NM', right_on='County', how='left')
ax = merged_gdf.plot(column='Count', cmap='Blues', legend=True)  # Added legend
ax.set_title('Combined County Counts (May-September)')  # Added title
ax.set_xlabel('Longitude')  # Added x-axis label
ax.set_ylabel('Latitude')  # Added y-axis label
plt.savefig("combined_county_counts.png")

# Create a point-based map with different colors for species
plt.figure(figsize=(45, 45))  # Increased figure size
combined_data['species_category'] = pd.Categorical(combined_data['ReportSpecies'])
ax = combined_data.plot(column='species_category', categorical=True, cmap='tab20', legend=True)  # Added legend
ax.set_title('Combined Species Distribution (May-September)')  # Added title
ax.set_xlabel('Longitude')  # Added x-axis label
ax.set_ylabel('Latitude')  # Added y-axis label
plt.savefig("combined_species_distribution.png")

In [0]:
dbutils.library.restartPython()

In [0]:
import os, glob, shutil, datetime

# 1) Define patterns for all outputs
patterns = [
    "*_county_counts.png",        # Monthly choropleth maps
    "*_species_distribution.png",      # Monthly point maps
    "combined_county_counts.png", # Combined choropleth
    "combined_species_distribution.png", # Combined point map
    "county_counts_by_month.xlsx"  # Excel summary
]

# 2) Collect matching files from current working directory
driver_dir = os.getcwd()
candidates = []
for pat in patterns:
    candidates.extend(glob.glob(os.path.join(driver_dir, pat)))

if not candidates:
    print("No matching files found.")
else:
    print(f"Found {len(candidates)} file(s):")
    for f in candidates:
        print(" -", os.path.basename(f))

# 3) Destination in DBFS FileStore
export_dir_dbfs = "dbfs:/FileStore/wbm_exports"
dbutils.fs.mkdirs(export_dir_dbfs)

# 4) Copy files to FileStore
published = []
for src in candidates:
    basename = os.path.basename(src)
    dest = f"{export_dir_dbfs}/{basename}"
    dbutils.fs.cp(f"file:{src}", dest)
    published.append(basename)

# 5) Generate clickable download links for each map and file
links = [
    f"<li><a href='/files/wbm_exports/{name}' download='{name}'>Download {name}</a></li>" for name in published
]
html = f"""
<h3>Download Links</h3>
<ul>
{''.join(links)}
</ul>
"""
displayHTML(html)

# 6) (Optional) Create ZIP for all files
zip_name = f"wbm_exports_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
zip_local = os.path.join("/tmp", zip_name)
tmp_stage = "/tmp/wbm_exports_stage"
os.makedirs(tmp_stage, exist_ok=True)
for src in candidates:
    shutil.copy2(src, os.path.join(tmp_stage, os.path.basename(src)))
shutil.make_archive(base_name=zip_local.replace(".zip", ""), format="zip", root_dir=tmp_stage)
zip_dest_dbfs = f"{export_dir_dbfs}/{zip_name}"
dbutils.fs.cp(f"file:{zip_local}", zip_dest_dbfs)

displayHTML(f"""
<h3>Download All as ZIP</h3>
<p><a href='/files{zip_name}' download='{zip_name}'>Download {zip_name}</a></p>
""")